# 03 - Baseline model

A linear regression on the grouped split, to set a reference point. I report train and test metrics together so any overfitting is visible, and I compare against a mean predictor so the linear model's lift is clear.

In [1]:
import sys
sys.path.append("..")

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from src.data_prep import load_raw, clean, split_by_profile
from src.features import make_preprocessor
from src.evaluate import regression_metrics, score_table

df = clean(load_raw())
X_train, X_test, y_train, y_test, groups_train = split_by_profile(df)
len(X_train), len(X_test)

(836542, 187944)

## Reference point: predict the mean

In [2]:
dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
regression_metrics(y_test, dummy.predict(X_test))

{'rmse': 19.571955365833137,
 'mae': 16.93016511955253,
 'r2': -0.07246446394395156}

Predicting the training mean for every row gives a test RMSE close to the target's spread and an R2 around zero (slightly negative because the held-out runs sit a little above the training mean). This is the floor any useful model must clear.

## Linear regression

In [3]:
linear = Pipeline([
    ("pre", make_preprocessor(scale=True)),
    ("model", LinearRegression()),
])
linear.fit(X_train, y_train)
score_table(linear, X_train, y_train, X_test, y_test)

,rmse,mae,r2
train,12.534,9.522,0.612
test,11.735,9.204,0.614


## Coefficients

In [4]:
import pandas as pd
from src.config import FEATURES

coefs = pd.Series(linear["model"].coef_, index=FEATURES).sort_values()
coefs.round(3)

u_d           -4.634
u_q           -4.068
i_d           -3.597
i_q           -2.809
coolant        5.785
ambient        6.559
motor_speed    7.157
dtype: float64

## What the baseline says

The linear model clears the mean-predictor floor by a wide margin, so there is real signal here. Train and test scores are close to each other, which tells me the model is not overfitting, if anything it is underfitting, since a single linear surface can't capture the way these quantities interact to heat the magnet. The coefficients (on standardised features, so they're comparable) line up with the EDA: ambient, motor speed and coolant push the temperature up, while the voltage and current components carry negative weights.

The gap between this and a perfect fit is the room nonlinear models should exploit. That is the next stage: Ridge and Lasso to check regularisation, then random forest and gradient boosting under grouped cross-validation.